# Normalizer retraining with masked entities (Kaggle)

Fine-tunes `google/flan-t5-base` on the masked training data from
`training/build_normalizer_data.py`. Places, dishes, cuisines and dietary words are typed
placeholders (`in Place1 got Diet1 food`), as the pipeline sends them at run time.

Settings are the cleaned-data run of `normalizer_training.ipynb` (3 epochs, lr 3e-5, batch 4,
best validation loss, seed 42), so the only change is the data.

**Before running**
1. **Data:** upload `train.csv` and `validation.csv` as a Kaggle Dataset (*Create → New Dataset*),
   then in this notebook *Add Input* → that dataset. Cell 2 finds them anywhere under `/kaggle/input`.
2. **Settings → Accelerator → GPU T4.** Cell 1 uses only the first GPU, so a T4 x2 session trains
   exactly like a single GPU. Avoid P100: recent PyTorch builds no longer support it.
3. **Settings → Internet → On**, to download `flan-t5-base` (and to push to the Hub at the end).

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one GPU: batch 4 stays batch 4 on a T4 x2 session
os.environ["WANDB_DISABLED"] = "true"

!pip install -q -U transformers datasets sentencepiece accelerate

import torch, transformers
print("transformers", transformers.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 1. Load the data

In [ ]:
import glob
import pandas as pd

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"{name} not found under /kaggle/input: add the dataset with Add Input")
    return hits[0]

train_df = pd.read_csv(find("train.csv"))
val_df = pd.read_csv(find("validation.csv"))
print(find("train.csv"), find("validation.csv"))
print("train", len(train_df), "| validation", len(val_df))
print(train_df.groupby(["source", "masked"]).size())
train_df[train_df["masked"]].head(8)[["singlish", "english"]]

## 2. Tokenize

The prefix must match `PREFIX` in `backend/src/normalizer/hf.py`.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

MODEL_NAME = "google/flan-t5-base"
PREFIX = "Normalize this Singlish into natural English: "

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def preprocess(batch):
    inputs = tokenizer([PREFIX + str(t) for t in batch["singlish"]], max_length=128, truncation=True)
    labels = tokenizer(text_target=[str(t) for t in batch["english"]], max_length=128, truncation=True)
    inputs["labels"] = labels["input_ids"]
    return inputs

def tokenize(df):
    ds = Dataset.from_pandas(df[["singlish", "english"]].reset_index(drop=True))
    return ds.map(preprocess, batched=True, remove_columns=ds.column_names)

train_tok, val_tok = tokenize(train_df), tokenize(val_df)
print(tokenizer.tokenize("in Place1 got Diet1 food"))  # each placeholder is 2-3 ordinary pieces

## 3. Train

Outputs go to `/kaggle/working`, which Kaggle keeps as the notebook's output.

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

OUT_DIR = "/kaggle/working/flan_t5_masked"

args = Seq2SeqTrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=3,
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    fp16=False,
    bf16=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=42,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model),
    processing_class=tokenizer,
)
trainer.train()

FINAL_DIR = f"{OUT_DIR}/final_model"
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print("saved to", FINAL_DIR)

## 4. Check the placeholders survive

Same decoding as the pipeline (`hf.py`): beam 4, no repeated trigrams, up to 128 new tokens.
`kept` counts the masked validation rows whose output still holds every placeholder of the
input; the pipeline throws away any output that doesn't.

In [ ]:
import re

PLACEHOLDER = re.compile(r"(?<!\w)(?:Place|Food|Diet)\d+(?!\w)", re.IGNORECASE)
model.eval()

def normalize(text):
    enc = tokenizer(PREFIX + text, return_tensors="pt", truncation=True, max_length=256).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=128, num_beams=4, no_repeat_ngram_size=3, early_stopping=True)
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

def keeps_all(src, out):
    found = {p.lower() for p in PLACEHOLDER.findall(out)}
    return all(p.lower() in found for p in PLACEHOLDER.findall(src))

masked_val = val_df[val_df["masked"]]
outputs = [normalize(s) for s in masked_val["singlish"]]
kept = sum(keeps_all(s, o) for s, o in zip(masked_val["singlish"], outputs))
print(f"placeholders kept: {kept}/{len(masked_val)}")

for text in ["in Place1 got Diet1 food or not?", "Food1 near Place1 got or not",
             "want Food1 for supper leh", "Place1 Food2 got?", "wah i very hungry leh, got any good place to makan?"]:
    print(f"{text!r:55} -> {normalize(text)!r}")

## 5. Save the model

**Option A, download:** zips the final model into `/kaggle/working`. Download it from the
notebook's *Output* panel (or the link the cell prints). Checkpoints are deleted first so the
output stays small.

**Option B, push to the Hub:** add your Hugging Face write token as a Kaggle secret named
`HF_TOKEN` (*Add-ons → Secrets*, then tick it for this notebook). Push to a *new* repo first, so
the deployed `makan-where-cs425/makan-flan-normalizer` stays as it is until the new model wins on
the eval. Then, locally:

```
MAKAN_NORMALIZER_MODEL=makan-where-cs425/makan-flan-normalizer-masked python eval/eval_normalizer.py
```

and compare with the current model (SEAT BLEU 54.5 / chrF 68.9; gold BLEU 12.5 / chrF 44.7,
46% unchanged).

In [ ]:
# Option A: zip for download
import shutil
from IPython.display import FileLink

for ckpt in glob.glob(f"{OUT_DIR}/checkpoint-*"):
    shutil.rmtree(ckpt)
zip_path = shutil.make_archive("/kaggle/working/flan_t5_masked", "zip", FINAL_DIR)
print(zip_path)
FileLink("flan_t5_masked.zip")

In [ ]:
# Option B: push to the Hub (needs the HF_TOKEN Kaggle secret)
# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import login
# login(token=UserSecretsClient().get_secret("HF_TOKEN"))
# model.push_to_hub("makan-where-cs425/makan-flan-normalizer-masked")
# tokenizer.push_to_hub("makan-where-cs425/makan-flan-normalizer-masked")